In [14]:
import pandas as pd

file_path = "/content/drive/MyDrive/Gen AI/Claim_Check/fraud_oracle.csv"

def validate_data(file_path):
    # Load data
    try:
        df = pd.read_csv(file_path)
        print(" Dataset successfully loaded!")
    except Exception as e:
        print(f" Error loading file: {e}")
        return

    # Check basic info
    print(f"\n--- Dataset Dimensions ---")
    print(f"Records: {df.shape[0]}, Columns: {df.shape[1]}")

    # Check for target class imbalance
    fraud_counts = df['FraudFound_P'].value_counts()
    fraud_pct = (fraud_counts[1] / df.shape[0]) * 100

    print(f"\n--- Class Distribution ---")
    print(f"Legitimate Claims: {fraud_counts[0]}")
    print(f"Fraudulent Claims: {fraud_counts[1]}")
    print(f"Fraud Percentage: {fraud_pct:.2f}%")

    # Check for missing values
    print("\n---Missing Values---")
    print(df.isnull().sum().sum())


    # Peek at first few rows
    print(f"\n--- Data Peek ---")
    print(df.head())

if __name__ == "__main__":
    validate_data('/content/drive/MyDrive/Gen AI/Claim_Check/fraud_oracle.csv')

 Dataset successfully loaded!

--- Dataset Dimensions ---
Records: 15420, Columns: 33

--- Class Distribution ---
Legitimate Claims: 14497
Fraudulent Claims: 923
Fraud Percentage: 5.99%

---Missing Values---
0

--- Data Peek ---
  Month  WeekOfMonth  DayOfWeek    Make AccidentArea DayOfWeekClaimed  \
0   Dec            5  Wednesday   Honda        Urban          Tuesday   
1   Jan            3  Wednesday   Honda        Urban           Monday   
2   Oct            5     Friday   Honda        Urban         Thursday   
3   Jun            2   Saturday  Toyota        Rural           Friday   
4   Jan            5     Monday   Honda        Urban          Tuesday   

  MonthClaimed  WeekOfMonthClaimed     Sex MaritalStatus  ...  AgeOfVehicle  \
0          Jan                   1  Female        Single  ...       3 years   
1          Jan                   4    Male        Single  ...       6 years   
2          Nov                   2    Male       Married  ...       7 years   
3          Jul  

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Load the data
df = pd.read_csv("/content/drive/MyDrive/Gen AI/Claim_Check/fraud_oracle.csv")

# 2. Drop unique identifier
if 'PolicyNumber' in df.columns:
    df = df.drop(columns=['PolicyNumber'])




In [3]:
### EXTRAAA

# Add this BEFORE your preprocessor step
df['VehiclePrice_to_Age'] = df['AgeOfVehicle'].map({'new': 1, '2 years': 2, '3 years': 3, '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7, 'more than 7': 8})
df['Price_Ratio'] = df['VehiclePrice'].map({'less than 20000': 1, '20000 to 29000': 2, '30000 to 39000': 3, '40000 to 59000': 4, '60000 to 69000': 5, 'more than 69000': 6}) / df['VehiclePrice_to_Age']

# Now your model has a "Suspicion Ratio" feature.
# Judges love it when you create domain-specific features!

In [4]:
# 3. Define Features and Target
X = df.drop(columns=['FraudFound_P'])
y = df['FraudFound_P']

# 4. Identify column types
# Numerical columns (we will scale these)
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
# Categorical columns (we will One-Hot Encode these)
cat_cols = X.select_dtypes(include=['object']).columns.tolist()


In [5]:
# ### EXTRAAA

# # Add this BEFORE your preprocessor step
# df['VehiclePrice_to_Age'] = df['AgeOfVehicle'].map({'new': 1, '2 years': 2, '3 years': 3, '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7, 'more than 7': 8})
# df['Price_Ratio'] = df['VehiclePrice'].map({'less than 20000': 1, '20000 to 29000': 2, '30000 to 39000': 3, '40000 to 59000': 4, '60000 to 69000': 5, 'more than 69000': 6}) / df['VehiclePrice_to_Age']

# # Now your model has a "Suspicion Ratio" feature.
# # Judges love it when you create domain-specific features!

In [6]:
# 5. Build the Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# 6. Stratified Split
# Stratify ensures the 6% fraud distribution is kept in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 7. Apply Transformation
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Training shape: {X_train_processed.shape}")
print(f"Test shape: {X_test_processed.shape}")
print("Preprocessing complete!")

Training shape: (12336, 149)
Test shape: (3084, 149)
Preprocessing complete!


In [7]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Random Forest Baseline
print("--- Training Random Forest ---")
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train_processed, y_train)
rf_preds = rf.predict(X_test_processed)
rf_probs = rf.predict_proba(X_test_processed)[:, 1]

print(f"Random Forest ROC-AUC: {roc_auc_score(y_test, rf_probs):.4f}")
print(classification_report(y_test, rf_preds))

# 2. XGBoost Baseline
print("\n--- Training XGBoost ---")
# scale_pos_weight is the ratio of negative to positive cases (roughly 15420/923 ≈ 16)
xgb = XGBClassifier(scale_pos_weight=16, random_state=42)
xgb.fit(X_train_processed, y_train)
xgb_preds = xgb.predict(X_test_processed)
xgb_probs = xgb.predict_proba(X_test_processed)[:, 1]

print(f"XGBoost ROC-AUC: {roc_auc_score(y_test, xgb_probs):.4f}")
print(classification_report(y_test, xgb_preds))

--- Training Random Forest ---
Random Forest ROC-AUC: 0.8166
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      2899
           1       1.00      0.01      0.01       185

    accuracy                           0.94      3084
   macro avg       0.97      0.50      0.49      3084
weighted avg       0.94      0.94      0.91      3084


--- Training XGBoost ---
XGBoost ROC-AUC: 0.8219
              precision    recall  f1-score   support

           0       0.96      0.91      0.93      2899
           1       0.19      0.33      0.24       185

    accuracy                           0.87      3084
   macro avg       0.57      0.62      0.59      3084
weighted avg       0.91      0.87      0.89      3084



In [8]:
## Tuning of XGBOOST

from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

# Defining the parameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'scale_pos_weight': [16] # Keep the 16:1 ratio we identified
}

print("--- Tuning XGBoost with GridSearchCV ---")
# Using ROC-AUC as the scoring metric
xgb_tuned = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

grid_search = GridSearchCV(estimator=xgb_tuned, param_grid=param_grid,
                           scoring='roc_auc', cv=3, verbose=1)

grid_search.fit(X_train_processed, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")

# Evaluate the best model
best_model = grid_search.best_estimator_
tuned_preds = best_model.predict(X_test_processed)
tuned_probs = best_model.predict_proba(X_test_processed)[:, 1]

from sklearn.metrics import classification_report, roc_auc_score
print(f"\nTuned XGBoost ROC-AUC: {roc_auc_score(y_test, tuned_probs):.4f}")
print(classification_report(y_test, tuned_preds))

--- Tuning XGBoost with GridSearchCV ---
Fitting 3 folds for each of 12 candidates, totalling 36 fits


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [20:50:10] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [20:50:11] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [20:50:12] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [20:50:14] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:


Best Parameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 100, 'scale_pos_weight': 16}

Tuned XGBoost ROC-AUC: 0.8349
              precision    recall  f1-score   support

           0       0.97      0.82      0.89      2899
           1       0.18      0.63      0.28       185

    accuracy                           0.81      3084
   macro avg       0.58      0.72      0.59      3084
weighted avg       0.92      0.81      0.85      3084



In [9]:
## MLP

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# --- Architecture ---
class FraudMLP(nn.Module):
    def __init__(self, input_dim):
        super(FraudMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

# --- Setup ---
model = FraudMLP(X_train_processed.shape[1])
# Using BCEWithLogitsLoss for numerical stability
pos_weight = torch.tensor([16.0])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)

# --- Training Loop ---
train_ds = TensorDataset(torch.tensor(X_train_processed, dtype=torch.float32),
                         torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

model.train()
for epoch in range(20):
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

# --- Evaluation ---
model.eval()
with torch.no_grad():
    test_probs = torch.sigmoid(model(torch.tensor(X_test_processed, dtype=torch.float32)))
    predictions = (test_probs > 0.5).float()

print("\n--- Deep Learning MLP Results ---")
print(f"ROC-AUC: {roc_auc_score(y_test, test_probs.numpy()):.4f}")
print(classification_report(y_test, predictions.numpy()))


--- Deep Learning MLP Results ---
ROC-AUC: 0.8310
              precision    recall  f1-score   support

           0       0.98      0.76      0.86      2899
           1       0.16      0.70      0.26       185

    accuracy                           0.76      3084
   macro avg       0.57      0.73      0.56      3084
weighted avg       0.93      0.76      0.82      3084



In [10]:
import shap
import torch

# 1. Initialize SHAP explainer for our PyTorch model
# We use a background sample from training data
background = torch.tensor(X_train_processed[:100], dtype=torch.float32)
explainer = shap.DeepExplainer(model, background)

# 2. Pick a "Fraud" sample from the test set to demonstrate
fraud_idx = (y_test == 1).idxmax() # Find index of a fraud case
sample = torch.tensor(X_test_processed[0:1], dtype=torch.float32) # Using first test row

# 3. Get SHAP values
shap_values = explainer.shap_values(sample)

# 4. Create the "Fraud Investigation Memo"
import numpy as np

# 1. Flatten the SHAP values to be safe
# DeepExplainer returns a list, the first element is the array of SHAP values
flat_shap_values = np.array(shap_values[0]).flatten()

# 2. Get feature names
feature_names = preprocessor.get_feature_names_out()

# 3. Find indices of the largest values (most positive contribution to fraud)
top_indices = np.argsort(flat_shap_values)[::-1][:3]

print("--- AI Auditor Report Generation ---")
print("Claim Flagged! Investigation Summary:")
for i in top_indices:
    factor = feature_names[i]
    impact = flat_shap_values[i]
    print(f"- Anomaly detected in: {factor} | Impact Score: {impact:.4f}")

# 4. Generate the narrative
reason1 = feature_names[top_indices[0]]
reason2 = feature_names[top_indices[1]]

print("\n--- GenAI Narrative ---")
print(f"Agent: 'This claim is flagged with high suspicion. "
      f"The model identified unusual patterns in '{reason1}' and '{reason2}'. "
      "Recommendation: Initiate a manual review of these specific claim parameters.'")

--- AI Auditor Report Generation ---
Claim Flagged! Investigation Summary:
- Anomaly detected in: cat__Fault_Third Party | Impact Score: 2.1203
- Anomaly detected in: cat__Fault_Policy Holder | Impact Score: 2.0676
- Anomaly detected in: cat__Make_Honda | Impact Score: 0.8093

--- GenAI Narrative ---
Agent: 'This claim is flagged with high suspicion. The model identified unusual patterns in 'cat__Fault_Third Party' and 'cat__Fault_Policy Holder'. Recommendation: Initiate a manual review of these specific claim parameters.'


In [11]:
import joblib
import torch

# 1. Save the model weights
torch.save(model.state_dict(), 'fraud_mlp.pt')

# 2. Save the preprocessor
joblib.dump(preprocessor, 'preprocessor.pkl')

# 3. Save the dataframe (with your new features)
df.to_csv('cleaned_fraud_data.csv', index=False)

print("✅ Files saved to the current environment!")

✅ Files saved to the current environment!


In [12]:
import shutil

# Move the files to your Drive folder
shutil.copy('fraud_mlp.pt', '/content/drive/MyDrive/Gen AI/Claim_Check/fraud_mlp.pt')
shutil.copy('preprocessor.pkl', '/content/drive/MyDrive/Gen AI/Claim_Check/preprocessor.pkl')
shutil.copy('cleaned_fraud_data.csv', '/content/drive/MyDrive/Gen AI/Claim_Check/cleaned_fraud_data.csv')

print("✅ Files safely moved to Google Drive!")

✅ Files safely moved to Google Drive!
